In [ ]:
import os
from dotenv import load_dotenv
import time


load_dotenv('../../../.env', override=True)


True

In [5]:
import logfire

try:
    logfire.configure(token=os.getenv('LOGFIRE_TOKEN'))
    logfire.instrument_pydantic_ai()
    print("✓ Logfire configured (cloud mode)")
    print("  Traces will be sent to Logfire homework project")
except Exception as e:
    print(f"Logfire setup issue: {e}")

✓ Logfire configured (cloud mode)
  Traces will be sent to Logfire homework project


In [6]:

import requests

url = "https://opentdb.com/api_category.php"
response = requests.get(url)
data = response.json()

print(f"Got {len(data['trivia_categories'])} categories")
print("\nFirst few:")
for cat in data['trivia_categories'][:5]:
    print(f"  {cat['id']}: {cat['name']}")

Logfire project URL: ]8;id=318711;https://logfire-eu.pydantic.dev/nirajanacharya/homework\https://logfire-eu.pydantic.dev/nirajanacharya/homework]8;;\

Got 24 categories

First few:
  9: General Knowledge
  10: Entertainment: Books
  11: Entertainment: Film
  12: Entertainment: Music
  13: Entertainment: Musicals & Theatres


In [7]:

def get_categories():
    url = "https://opentdb.com/api_category.php"
    data = requests.get(url).json()
    
    result = []
    for cat in data['trivia_categories']:
        result.append(f"{cat['id']}: {cat['name']}")
    
    return "\n".join(result)


categories = get_categories()
print(categories)

9: General Knowledge
10: Entertainment: Books
11: Entertainment: Film
12: Entertainment: Music
13: Entertainment: Musicals & Theatres
14: Entertainment: Television
15: Entertainment: Video Games
16: Entertainment: Board Games
17: Science & Nature
18: Science: Computers
19: Science: Mathematics
20: Mythology
21: Sports
22: Geography
23: History
24: Politics
25: Art
26: Celebrities
27: Animals
28: Vehicles
29: Entertainment: Comics
30: Science: Gadgets
31: Entertainment: Japanese Anime & Manga
32: Entertainment: Cartoon & Animations


In [8]:

import html

def get_questions(amount, category, difficulty):
    params = {
        'amount': amount,
        'category': category,
        'difficulty': difficulty,
        'type': 'multiple'
    }
    url = "https://opentdb.com/api.php"
    data = requests.get(url, params=params).json()
    
    result = []
    for i, q in enumerate(data['results'], 1):
 
        question = html.unescape(q['question'])
        correct = html.unescape(q['correct_answer'])
        wrong = [html.unescape(a) for a in q['incorrect_answers']]
        
        result.append(f"Q{i}: {question}")
        result.append(f"Correct: {correct}")
        result.append(f"Wrong: {', '.join(wrong)}\n")
    
    return "\n".join(result)


questions = get_questions(2, 17, "easy")
print(questions)

Q1: The medical term for the belly button is which of the following?
Correct: Umbilicus
Wrong: Nevus, Nares, Paxillus

Q2: Which element has the highest melting point?
Correct: Carbon
Wrong: Tungsten, Platinum, Osmium



In [9]:
from pydantic_ai import Agent

class TriviaTools:
    def get_categories(self):
        url = "https://opentdb.com/api_category.php"
        data = requests.get(url).json()
        result = []
        for cat in data['trivia_categories']:
            result.append(f"{cat['id']}: {cat['name']}")
        return "\n".join(result)
    
    def get_questions(self, amount: int, category: int, difficulty: str):
        params = {
            'amount': amount,
            'category': category,
            'difficulty': difficulty,
            'type': 'multiple'
        }
        url = "https://opentdb.com/api.php"
        data = requests.get(url, params=params).json()
        
        result = []
        for i, q in enumerate(data['results'], 1):
            question = html.unescape(q['question'])
            correct = html.unescape(q['correct_answer'])
            wrong = [html.unescape(a) for a in q['incorrect_answers']]
            result.append(f"Q{i}: {question}")
            result.append(f"Correct: {correct}")
            result.append(f"Wrong: {', '.join(wrong)}\n")
        return "\n".join(result)

tools = TriviaTools()

instructions = """You are a trivia quizmaster. When asked to play trivia:
1. Use get_categories to see available categories
2. Use get_questions to fetch questions
3. Ask questions one at a time with multiple choice answers
4. Wait for player's answer 
5. Explain why the correct answer is right (add interesting facts!)
6. Give final score at the end
"""

agent = Agent(
    "openai:gpt-4o-mini",
    tools=[tools.get_categories, tools.get_questions],
    system_prompt=instructions
)

print("Agent created!")

Agent created!


In [10]:

result = await agent.run("Let's play trivia! Give me 2 easy science questions.")
print(result.output)

06:29:51.319 agent run
06:29:51.327   chat gpt-4o-mini
06:29:53.037   running 1 tool
06:29:53.039     running tool: get_categories
06:29:54.457   chat gpt-4o-mini
06:29:56.743   running 1 tool
06:29:56.744     running tool: get_questions
06:29:58.247   chat gpt-4o-mini
Great! Let's start the trivia with your first easy science question:

### Question 1:
**Which element has the highest melting point?**
A) Tungsten  
B) Platinum  
C) Carbon  
D) Osmium  

What's your answer?


In [11]:
async def run_trivia(prompt):
    messages = []
    
    while True:
        result = await agent.run(prompt, message_history=messages)
        print("\n" + result.output)
        messages = result.all_messages()
        
        answer = input("\nYou: ")
        if not answer or answer.lower() == 'stop':
            break
        prompt = answer
    
    return messages


In [12]:
async def run_trivia_grouped(prompt):
    with logfire.span('trivia_session'):
        messages = []
        
        while True:
            result = await agent.run(prompt, message_history=messages)
            print("\n" + result.output)
            messages = result.all_messages()
            
            answer = input("\nYou: ")
            if not answer or answer.lower() == 'stop':
                break
            prompt = answer
        
        return messages



In [13]:

from logfire.query_client import LogfireQueryClient

client = LogfireQueryClient(read_token='pylf_v1_eu_RvDzbzsBfKl6LCTncrW9bkBc9h9gwvP01p2Sr5l34Bk0')
print("Query client ready")

Query client ready


In [14]:

async def test_trivia_session():
    """Quick automated session (no user input needed)"""
    with logfire.span('trivia_session'):
       
        result = await agent.run("Give me 1 easy science question")
        print("Test session complete!")
        return result.output


await test_trivia_session()

06:30:00.534 trivia_session
06:30:00.535   agent run
06:30:00.536     chat gpt-4o-mini


06:30:01.483     running 1 tool
06:30:01.483       running tool: get_categories
06:30:02.543     chat gpt-4o-mini
06:30:03.516     running 1 tool
06:30:03.517       running tool: get_questions
06:30:04.641     chat gpt-4o-mini
Test session complete!


"Here's your easy science question:\n\n**Dry ice is the solid form of what substance?**\nA) Nitrogen  \nB) Ammonia  \nC) Oxygen  \nD) Carbon dioxide  \n\nWhat's your answer?"

In [ ]:

def to_rows(result):
    cols = {col['name']: col['values'] for col in result['columns']}
    if not cols:
        return []
    n = len(next(iter(cols.values())))
    return [{k: v[i] for k, v in cols.items()} for i in range(n)]

def safe_query(query, max_retries=3, delay=5):
    """Execute query with rate limit retry logic"""
    for attempt in range(max_retries):
        try:
            return client.query_json(sql=query)
        except AssertionError as e:
            if 'Rate limit exceeded' in str(e) and attempt < max_retries - 1:
                print(f"⏳ Rate limit hit, waiting {delay}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(delay)
            else:
                raise
    return None

def get_trace_ids():
    query = '''
    SELECT DISTINCT trace_id
    FROM records
    WHERE span_name = 'trivia_session'
    '''
    result = safe_query(query)
    return [row['trace_id'] for row in to_rows(result)]

trace_ids = get_trace_ids()
print(f"Found {len(trace_ids)} trivia sessions")
for tid in trace_ids[:3]:
    print(f"  {tid}")


Found 4 trivia sessions
  019cb1055aa35a837de18b5fff549f9b
  019cb116c65fc487cfd54f4ffe6c52f5
  019cb1265d23462a067b31c383b1cc7d


In [16]:

import json
debug = client.query_json(sql="""
    SELECT span_name, attributes
    FROM records
    WHERE span_name = 'chat gpt-4o-mini'
    LIMIT 1
""")
rows = to_rows(debug)
if rows:
    print("Chat span attributes:")
    print(json.dumps(rows[0]['attributes'], indent=2))


Chat span attributes:
{
  "gen_ai.input.messages": [
    {
      "role": "system",
      "parts": [
        {
          "type": "text",
          "content": "You are a trivia quizmaster. When asked to play trivia:\n1. Use get_categories to see available categories\n2. Use get_questions to fetch questions\n3. Ask questions one at a time with multiple choice answers\n4. Wait for player's answer \n5. Explain why the correct answer is right (add interesting facts!)\n6. Give final score at the end\n"
        }
      ]
    },
    {
      "role": "user",
      "parts": [
        {
          "type": "text",
          "content": "Let's play trivia! Give me 2 easy science questions."
        }
      ]
    },
    {
      "role": "assistant",
      "parts": [
        {
          "type": "tool_call",
          "id": "call_OMzL1C8z3biMLNa9dNOsrykP",
          "name": "get_categories",
          "arguments": "{}"
        }
      ],
      "finish_reason": "tool_call"
    },
    {
      "role": "user",

**Answer:** Query for records where `span_name = 'trivia_session'` and get distinct trace_ids.

In [ ]:
def get_sessions():
    query = '''
    SELECT 
        trace_id,
        start_timestamp,
        span_id,
        message
    FROM records
    WHERE span_name = 'trivia_session'
    ORDER BY start_timestamp DESC
    '''
    result = safe_query(query)
    return to_rows(result)

sessions = get_sessions()
print(f"Found {len(sessions)} sessions:\n")
for s in sessions[:3]:
    print(f"Trace: {s['trace_id'][:16]}...")
    print(f"Time: {s['start_timestamp']}")
    print(f"Span: {s['span_id']}")
    print()


Found 5 sessions:

Trace: 019cb127c8f62efb...
Time: 2026-03-03T00:45:00.534121Z
Span: 11abb634e5874389

Trace: 019cb1265d23462a...
Time: 2026-03-03T00:43:27.395591Z
Span: d4f96583137f0ff8

Trace: 019cb116c65fc487...
Time: 2026-03-03T00:26:25.759331Z
Span: 5d024cfd93d07374



In [ ]:
def get_token_usage(trace_id):
    
    query = f"""
    SELECT attributes
    FROM records
    WHERE trace_id = '{trace_id}'
    AND span_name = 'chat gpt-4o-mini'
    """
    result = safe_query(query)
    rows = to_rows(result)
    
    total_input = 0
    total_output = 0
    for row in rows:
        attrs = row.get('attributes', {}) or {}
        total_input += attrs.get('gen_ai.usage.input_tokens', 0) or 0
        total_output += attrs.get('gen_ai.usage.output_tokens', 0) or 0
    
    return {'input': int(total_input), 'output': int(total_output)}


if trace_ids:
    usage = get_token_usage(trace_ids[0])
    print(f"Usage for trace {trace_ids[0][:16]}...")
    print(f"  Input: {usage['input']} tokens")
    print(f"  Output: {usage['output']} tokens")


Usage for trace 019cb1055aa35a83...
  Input: 781 tokens
  Output: 77 tokens


In [19]:

PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o": {"input": 5.00, "output": 15.00},
}

def calculate_cost(model, input_tokens, output_tokens):
    if model not in PRICES:
        raise ValueError(f"Unknown model: {model}")
    
    prices = PRICES[model]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    
    return input_cost + output_cost


cost = calculate_cost("gpt-4o-mini", 500, 200)
print(f"Cost for 500 input + 200 output tokens: ${cost:.6f}")

Cost for 500 input + 200 output tokens: $0.000195


In [20]:

def get_trace_cost(trace_id, model="gpt-4o-mini"):
    usage = get_token_usage(trace_id)
    return calculate_cost(model, usage['input'], usage['output'])


if trace_ids:
    cost = get_trace_cost(trace_ids[0])
    print(f"Session cost: ${cost:.6f}")

Session cost: $0.000163


In [ ]:
import questionary

def ask_feedback():
    result = questionary.select(
        "How was the trivia session?",
        choices=["👍 Good", "👎 Bad", "Skip"],
    ).ask()

    if result is None or result == "Skip":
        return None

    return 1 if "Good" in result else -1


In [ ]:

async def run_with_feedback(prompt):
    with logfire.span('trivia_session'):
        messages = []
        
        while True:
            result = await agent.run(prompt, message_history=messages)
            print("\n" + result.output)
            messages = result.all_messages()
            
            answer = input("\nYou: ")
            if not answer or answer.lower() == 'stop':
                break
            prompt = answer
        
        # Ask for feedback at the end
        feedback = ask_feedback()
        if feedback is not None:
            logfire.info("user_feedback", rating=feedback)
            print(f"\nThanks for the feedback!")
        
        return messages


In [ ]:
from collections import Counter

def get_feedback_stats():
    query = """
    SELECT attributes
    FROM records
    WHERE message = 'user_feedback'
    """
    result = safe_query(query)
    rows = to_rows(result)
    
    counts = Counter()
    for row in rows:
        attrs = row.get('attributes', {}) or {}
        rating = attrs.get('rating', 'unknown')
        counts[rating] += 1
    
    return [{'rating': k, 'count': v} for k, v in counts.most_common()]

stats = get_feedback_stats()
if stats:
    print("Feedback stats:")
    for s in stats:
        print(f"  {s['rating']}: {s['count']}")
else:
    print("No feedback yet - run a session with run_with_feedback() first!")


AssertionError: b'{"detail":"Rate limit exceeded for organization nirajanacharya: per minute limit reached."}'

In [ ]:

total_input, total_output = 0, 0
for tid in trace_ids:
    u = get_token_usage(tid)
    total_input += u['input']
    total_output += u['output']
    print(f"  trace {tid[:16]}... input={u['input']} output={u['output']}")

total_tokens = total_input + total_output
total_cost = sum(get_trace_cost(tid) for tid in trace_ids)

print(f"\nQ4 - Total tokens across all sessions: {total_tokens}")
print(f"Q6 - Total cost: ${total_cost:.6f}")


## 📝 Homework 5 - Answers Summary

In [ ]:
# HOMEWORK ANSWERS SUMMARY

print("=" * 60)
print("HOMEWORK 5 - ANSWERS SUMMARY")
print("=" * 60)

print("\n📝 Question 1: Create and Run the Agent")
print("Answer: get_trivia")
print("Explanation: The main function to run trivia is get_trivia (or run_trivia variants)")

print("\n📝 Question 2: Set Up Monitoring")
print("Answer: pydantic_ai.agent")
print("Explanation: We instrument with logfire.instrument_pydantic_ai()")

print("\n📝 Question 3: Play a Full Session")
print(f"Answer: {len(trace_ids)} sessions")
print("Explanation: Count of trivia_session spans from Logfire")

print(f"\n📝 Question 4: Total Tokens Across All Sessions")
total_input, total_output = 0, 0
for tid in trace_ids:
    u = get_token_usage(tid)
    total_input += u['input']
    total_output += u['output']

total_tokens = total_input + total_output
print(f"Total tokens: {total_tokens}")
if total_tokens < 2000:
    print("Answer: Less than 2,000")
elif total_tokens < 10000:
    print("Answer: 2,000 - 10,000")
elif total_tokens < 20000:
    print("Answer: 10,000 - 20,000")
else:
    print("Answer: More than 20,000")

print("\n📝 Question 5: Get Trace IDs with SQL")
print("Answer: Query for records where `span_name = 'trivia_session'` and get distinct trace_ids.")

print(f"\n📝 Question 6: Total Cost")
total_cost = sum(get_trace_cost(tid) for tid in trace_ids)
print(f"Total cost: ${total_cost:.6f}")
if total_cost < 0.01:
    print("Answer: Less than $0.01")
elif total_cost < 0.05:
    print("Answer: $0.01 - $0.05")
elif total_cost < 0.10:
    print("Answer: $0.05 - $0.10")
else:
    print("Answer: More than $0.10")

print("\n📝 Question 7: Feedback Tracking")
print("Answer: By using logfire.attach_context() with the session context")
print("Explanation: In run_with_feedback(), we use logfire.span('trivia_session')")
print("and logfire.info('user_feedback', rating=feedback) within that span context")

print("\n" + "=" * 60)
